In [3]:
import asyncio

async def hello():
    await asyncio.sleep(2)
    print("hello")

print("before")

await hello() # Do this in Notebook

print("after")

# asyncio.run(hello()) # Do this in .py file


before
hello
after


## 2. async and await

`async` tells Python that a function can perform asynchronous operations.

`await` tells Python:

"Pause this coroutine until this asynchronous operation is ready, but allow the event loop to do other work."

`await` can only be used inside an `async` function.

In [4]:
import asyncio

async def fetch_data():
    print("Fetching Data...")

    await asyncio.sleep(2)

    print ("Data Recieved !")
    return "Some data"

async def main():
    data = await fetch_data()
    print(data)

await main() 

Fetching Data...
Data Recieved !
Some data


In [5]:
import asyncio
import time


async def task_a():
    print("Task A started")
    await asyncio.sleep(2)
    print("Task A finished")


async def task_b():
    print("Task B started")
    await asyncio.sleep(1)
    print("Task B finished")


async def main():
    await task_a()
    await task_b()


start = time.perf_counter()

await main()

print(f"Time taken: {time.perf_counter() - start:.2f} seconds")

Task A started
Task A finished
Task B started
Task B finished
Time taken: 3.03 seconds


## 4. Concurrency with asyncio.gather()

So far, we waited for Task A to finish before starting Task B.

But if the tasks are independent, we can run them concurrently.

`asyncio.gather()` schedules multiple coroutines and waits for all of them to complete.

This is particularly useful for I/O-bound operations such as:

- API calls
- Database queries
- LLM requests
- File/network operations

In [6]:
import asyncio
import time


async def task_a():
    print("Task A started")
    await asyncio.sleep(2)
    print("Task A finished")

    return "Result A"


async def task_b():
    print("Task B started")
    await asyncio.sleep(1)
    print("Task B finished")

    return "Result B"


async def main():
    results = await asyncio.gather(
        task_a(),
        task_b(),
    )

    print(results)


start = time.perf_counter()

await main()

# asyncio.run(main())

print(f"Time taken: {time.perf_counter() - start:.2f} seconds")

Task A started
Task B started
Task B finished
Task A finished
['Result A', 'Result B']
Time taken: 2.01 seconds


## 5. The Event Loop

The event loop is the engine behind asyncio.

It keeps track of asynchronous tasks and resumes them when they are ready to make progress.

When one task is waiting for I/O, the event loop can work on another task. You normally don't need to manually manage the event loop.

For standalone Python programs, `asyncio.run()` is the common entry point. But a Jupyter/VS Code notebook kernel is already running inside its own asyncio event loop (that's how the kernel manages cells under the hood). Notebooks support top-level await for exactly this reason — there's already a loop running, so you hook into it instead of starting a new one.

`asyncio.run()` is for plain .py scripts run from the terminal, where no loop exists yet and something has to create one — that's the if `__name__ == "__main__": asyncio.run(main())` pattern.

## 7. Async Generators and async for

An async generator can produce values asynchronously over time.

We use:

    async def
    yield

to create an async generator.

We consume it using:

    async for

This is particularly useful for streaming responses from LLMs or other APIs.

In [7]:
import asyncio

async def generate_numbers():
    for i in range(5):
        await asyncio.sleep(1)
        yield i

async def main():
    async for number in generate_numbers():
        print(number)

await main()

0
1
2
3
4


In [8]:
import asyncio
import httpx

async def fetch_post():

    async with httpx.AsyncClient() as client:
        response = await client.get("https://jsonplaceholder.typicode.com/posts/1")
        return response.json()

async def main():
    post = await fetch_post()
    print(post)

await main()

{'userId': 1, 'id': 1, 'title': 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit', 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}


In [12]:
import asyncio
import httpx

async def fetch_user(client):
    response = await client.get("https://jsonplaceholder.typicode.com/users/1")
    return response.json()

async def fetch_post(client):
    response = await client.get("https://jsonplaceholder.typicode.com/posts/1")
    return response.json()

async def fetch_todos(client):
    response = await client.get("https://jsonplaceholder.typicode.com/todos/1")
    return response.json()

async def main():

    async with httpx.AsyncClient() as client:
        user, post, todos = await asyncio.gather(
            fetch_user(client),
            fetch_post(client),
            fetch_todos(client)
        )
        

    print("User:", user["name"])
    print("Post:", post["title"])
    print("Todos:", todos["title"])

await main()

User: Leanne Graham
Post: sunt aut facere repellat provident occaecati excepturi optio reprehenderit
Todos: delectus aut autem
